# 04 · Embeddings and search
Turns every chunk into an embedding (a list of 384 numbers that represents its meaning) using the
`BAAI/bge-small-en-v1.5` model, stores them in ChromaDB, and runs the first semantic searches.
Saves the vector database to `data/chroma`.

In [1]:
# ===== CELL 1: Load the chunks and the embedding model =====
# The embedding model turns text into 384 numbers that represent its meaning.

import time                                             # time: to measure how long embedding takes
import pandas as pd                                     # pandas: tables
import chromadb                                         # chromadb: the vector database that stores and searches embeddings
from sentence_transformers import SentenceTransformer   # SentenceTransformer: loads and runs embedding models

chunks = pd.read_json("../data/processed/chunks.jsonl", lines=True)   # Load the chunks saved by 03_chunk
print("Chunks:", len(chunks))                           # How many there are (should be 6,601)

MODEL_NAME = "BAAI/bge-small-en-v1.5"                   # The embedding model (downloaded from Hugging Face the first time)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # This model works best when QUESTIONS start with this instruction

model = SentenceTransformer(MODEL_NAME)                 # Load the model (downloads about 130 MB the first time)
print("Model loaded. Vector size:", model.get_sentence_embedding_dimension())   # Numbers per embedding (384 for this model)

Chunks: 6601


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded. Vector size: 384


C:\Users\lenovo\AppData\Local\Temp\ipykernel_15328\964722337.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Vector size:", model.get_sentence_embedding_dimension())   # Numbers per embedding (384 for this model)


In [2]:
# ===== CELL 2: Embed all the chunks =====
# Runs every chunk through the model. Takes a few minutes on a laptop CPU.

start = time.time()                          # Remember the start time (in seconds)
embeddings = model.encode(                   # Turn the texts into embeddings
    chunks["embed_text"].tolist(),           # The texts: every chunk's search text (header + passage)
    batch_size=32,                           # Process 32 chunks at a time (faster than one by one)
    show_progress_bar=True,                  # Show a progress bar while it works
    normalize_embeddings=True,               # Scale every vector to the same length, so comparisons measure meaning, not text length
)
print(f"Done in {time.time() - start:.0f} seconds. Shape: {embeddings.shape}")   # Time taken, and the shape: (number of chunks, 384)

Batches:   0%|          | 0/207 [00:00<?, ?it/s]

Done in 1357 seconds. Shape: (6601, 384)


In [3]:
# ===== CELL 3: Store everything in ChromaDB =====
# Saves embeddings, texts and details to a database on disk, so we only embed once.

client = chromadb.PersistentClient(path="../data/chroma")   # Open (or create) a database saved in the data/chroma folder
COLLECTION = "aspnetcore_docs_bge_small"                    # Collection name (like a table); includes the model name so we can add others later

try:                                         # Try to...
    client.delete_collection(COLLECTION)     # ...delete the collection if it already exists (so rerunning starts fresh)
except Exception:                            # If it doesn't exist yet, that raises an error...
    pass                                     # ...which we simply ignore
collection = client.create_collection(COLLECTION)   # Create a new, empty collection

meta_cols = ["doc_id", "area", "title", "section", "url"]   # The extra details stored with each chunk
BATCH = 1000                                 # Add 1,000 chunks at a time (the database limits how many per call)
for i in range(0, len(chunks), BATCH):       # i = 0, 1000, 2000, ... up to the number of chunks
    part = chunks.iloc[i:i + BATCH]          # Take rows i to i+1000
    collection.add(                          # Add them to the database
        ids=part["chunk_id"].tolist(),                  # A unique ID for each chunk
        embeddings=embeddings[i:i + BATCH].tolist(),    # The matching embeddings (we supply our own, so we control the model)
        documents=part["text"].tolist(),                # The passage text (returned with search results)
        metadatas=part[meta_cols].to_dict("records"),   # The details, one dictionary per chunk
    )

print("Stored chunks:", collection.count())  # How many chunks the database now holds

Stored chunks: 6601


In [4]:
# ===== CELL 4: A search function =====
# search(): turns a question into an embedding and finds the closest chunks.
# show():   prints the results in a readable way.

def search(query, k=5, area=None):           # query: the question; k: how many results; area: optionally "blazor", "mvc" or "fundamentals"
    q = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)   # Embed the question, with the instruction this model expects
    res = collection.query(                  # Ask the database for the closest chunks
        query_embeddings=q.tolist(),         # The question's embedding
        n_results=k,                         # How many results to return
        where={"area": area} if area else None,   # Filter by area if one was given, otherwise search everything
    )
    hits = []                                # The results go here
    for cid, text, meta, dist in zip(res["ids"][0], res["documents"][0],
                                     res["metadatas"][0], res["distances"][0]):   # Walk through the 4 result lists together; [0] because we asked one question
        hits.append({                        # Save one result
            "chunk_id": cid,                 # Which chunk
            "score": round(1 - dist / 2, 3), # Chroma gives a distance (smaller = closer); this turns it into a similarity (1.0 = identical meaning)
            "where": meta["title"] + (f" > {meta['section']}" if meta["section"] else ""),   # "Article > Section", for display
            "text": text,                    # The chunk's passage
        })
    return hits                              # Return the list of results

def show(query, k=3):                        # Print the top k results for a question
    print(f"Q: {query}\n")                   # The question, then a blank line
    for h in search(query, k):               # For each result...
        print(f"  {h['score']:.3f} | {h['where']}")                       # ...its score and where it comes from...
        print(f"         {h['text'][:200].replace(chr(10), ' ')}...\n")   # ...and its first 200 characters, line breaks turned into spaces
    print("=" * 100)                         # A separator line

In [6]:
# ===== CELL 5: Your first searches =====
# None of these questions uses the exact words of the answer,
# so good results mean the search understands meaning.

show("How do I register a service so a new instance is created per HTTP request?")   # Fundamentals: should find the "scoped" lifetime
show("What does the @key directive attribute do in Blazor?")                       # Blazor
show("How do I apply a filter to all controllers globally?")                       # MVC
show("How do I read a value from appsettings.json?")                               # Fundamentals: configuration

Q: How do I register a service so a new instance is created per HTTP request?

  0.745 | HTTP requests with IHttpClientFactory - ASP.NET Core > Review consumption patterns > Typed clients
         The following code calls AddHttpClient in the _Program.cs_ file to register the `GitHubService` typed client class:  [Code sample: Program.cs]  The typed client is registered as transient with DI. In ...

  0.745 | Prerender ASP.NET Core Razor components > Client-side services fail to resolve during prerendering > Create a service abstraction
         If a different service implementation is needed on the server, create a service abstraction and create implementations for the service in the server and client projects. Register the services in each ...

  0.742 | ASP.NET Core Blazor dependency injection > Register common services
         If one or more common services are required client- and server-side, you can place the common service registrations in a method client-side and call the meth

In [7]:
# ===== CELL 6: Try your own =====
# Ask something you know the answer to, then check whether the top results are right.

show("What is dependency injection?")   # Replace the text between the quotes with your question

Q: What is dependency injection?

  0.789 | Dependency injection in ASP.NET Core > Overview of dependency injection
         A *dependency* is an object that another object depends on. Consider the following `MyDependency` class with a `WriteMessage` method:  ```csharp public class MyDependency {     public void WriteMessag...

  0.741 | ASP.NET Core Blazor dependency injection
         By Rainer Stropek and Mike Rousos  This article explains how Blazor apps can inject services into components.  Dependency injection (DI) is a technique for accessing services configured in a central l...

  0.739 | Dependency injection in ASP.NET Core > Overview of dependency injection
         DI addresses these problems through:  * The use of an interface or base class to abstract the dependency implementation. * Registration of the dependency in a *service container*, also called a *DI co...

